<a href="https://colab.research.google.com/github/ZIEYA/ColabLibrarly/blob/main/OpenDeepResearch_V01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install -U -q open-deep-research
! pip install -U langchain-google-genai # Install the missing package

# 環境変数の定義
from google.colab import userdata
import os
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["GOOGLE_PROJECT_ID"] = userdata.get("GOOGLE_PROJECT_ID")

from IPython.display import display
from langgraph.types import Command
from langgraph.checkpoint.memory import MemorySaver
from open_deep_research.graph import builder

# メモリとグラフの初期化:
memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

# スレッドパラメータの定義
import uuid
thread = {"configurable": {"thread_id": str(uuid.uuid4()),
                           "search_api": "tavily",
                           "planner_provider": "google_genai",
                           "planner_model": "gemini-2.5-flash-preview-04-17",
                           "writer_provider": "google_genai",
                           "writer_model": "gemini-2.0-flash",
                           "max_search_depth": 1,
                           }}

from IPython.display import Markdown
# Create a topic
topic = "ガンダムシリーズおける宇宙世紀の年表を作成して欲しい。その際、シリーズ各作品との関連も示して欲しい。なお出力は全て日本語で行うこと"

# 割り込みが発生するまで Graph を非同期実行
async for event in graph.astream({"topic":topic,}, thread, stream_mode="updates"):
    if '__interrupt__' in event:
        interrupt_value = event['__interrupt__'][0].value
        display(Markdown(interrupt_value)) #レポートプランを表示する

# Graph の実行を再開し進行状況を表示
async for event in graph.astream(Command(resume=True), thread, stream_mode="updates"):
    print(event) # 処理状況を表示する
    print("\n")

# 最終レポートの表示
final_state = graph.get_state(thread)
report = final_state.values.get('final_report')
Markdown(report)  #最終的なレポートをMarkdown表示する
